# Clasificación con Regresión Logística Multinomial: Palmer Penguins

Este notebook es la **continuación directa** del análisis exploratorio de Palmer
Penguins. Allá describimos los datos; aquí construimos un modelo que **predice**
la especie de un pingüino a partir de sus medidas corporales.

Es la primera vez en el curso que un modelo predice una **categoría** en lugar de
un número, así que vale la pena decir desde el principio qué cambia y qué no.

## De dónde venimos

En el capítulo de fundamentos establecimos que armar un modelo de Machine
Learning siempre es elegir **tres piezas**:

| Pieza | Regresión lineal | Regresión logística (2 clases) | **Multinomial (3 clases)** |
|---|---|---|---|
| **Familia de funciones** | Combinación lineal | **Sigmoide** de una combinación lineal | **Softmax** de *tres* combinaciones lineales |
| **Función de costo** | Error cuadrático | Log-loss (entropía cruzada) | **La misma**, extendida a 3 términos |
| **Optimizador** | Fórmula cerrada o descenso en gradiente | Descenso en gradiente | **El mismo** |

Léanla en vertical: de las tres casillas, **el optimizador no cambia nunca**, y
la función de costo es la misma idea. Lo único genuinamente nuevo en este
notebook es la primera casilla. Todo lo demás ya lo vieron.

## Objetivos

1. Entender el **softmax** como la generalización natural de la sigmoide
2. Distinguir las **cuatro escalas** en las que vive un clasificador: logit,
   momios, probabilidad y etiqueta
3. Ajustar e interpretar una logística multinomial con `scikit-learn`
4. Evaluar con matriz de confusión, precision/recall y log-loss —
   y ver por qué el *accuracy* solo no basta
5. Escribir el **descenso en gradiente a mano** y comprobar que llega
   exactamente a lo mismo que `sklearn`
6. Reconocer **separación perfecta** y entender para qué sirve la regularización

## Lo que este notebook NO hace

Los pingüinos no tienen una decisión de negocio asociada: no hay un costo por
equivocarse, así que no hay un umbral que valga la pena mover. Esa parte —
matriz de costos, ajuste del umbral, curvas de ganancia — se queda en los
ejemplos de churn y fraude del capítulo. Aquí el objetivo es **ver la mecánica
funcionando sobre datos reales que ya conocen**.

## 0. Configuración del entorno

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    log_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Mismo estilo que el notebook de EDA
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

# Mismos colores por especie que en el EDA
COLORES_ESPECIE = {
    "Adelie": "#E69F00",
    "Chinstrap": "#009E73",
    "Gentoo": "#56B4E9",
}

# Para pedir "sin regularización" hay que pasar C=np.inf. scikit-learn lo
# traduce internamente a penalty=None y avisa que ignora C; el aviso es
# esperado y no indica ningún problema.
warnings.filterwarnings("ignore", message="Setting penalty=None")

SEMILLA = 42
np.random.seed(SEMILLA)

---

## 1. De la sigmoide al softmax

### El problema

En el capítulo, la regresión logística respondía preguntas de **sí o no**:
¿se va el cliente? ¿es fraude? Una sola probabilidad $p$, y su complemento
$1 - p$ salía gratis.

Aquí la pregunta tiene **tres respuestas posibles**: Adelie, Chinstrap o Gentoo.
Necesitamos tres probabilidades que sumen 1, y una sola sigmoide no puede
producirlas.

### La construcción, en tres pasos

La solución es la más obvia posible: **una combinación lineal por clase**.

**Paso 1 — un score por clase.** En lugar de un solo $z$, calculamos tres:

$$z_{\text{Adelie}} = \beta_{0}^{A} + \beta_{1}^{A}x_1 + \beta_{2}^{A}x_2 \qquad
z_{\text{Chinstrap}} = \beta_{0}^{C} + \dots \qquad
z_{\text{Gentoo}} = \beta_{0}^{G} + \dots$$

Cada clase tiene su **propio juego de coeficientes**. El score $z_k$ dice
qué tanto "vota" el modelo por la clase $k$. Puede ser cualquier número real,
positivo o negativo.

**Paso 2 — exponenciar.** $e^{z_k}$ convierte cada score en un número
**positivo**, y preserva el orden: más score, más peso.

**Paso 3 — normalizar.** Dividimos entre la suma para que los tres sumen 1:

$$P(y = k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

Eso es el **softmax**. No hay más.

### Por qué es la misma función de antes

Con $K = 2$ clases, el softmax **es** la sigmoide. Basta dividir arriba y abajo
entre $e^{z_1}$:

$$P(y=1) = \frac{e^{z_1}}{e^{z_1} + e^{z_2}}
= \frac{1}{1 + e^{-(z_1 - z_2)}} = \sigma(z_1 - z_2)$$

La sigmoide era el softmax de dos clases todo este tiempo. Comprobémoslo con
números:

In [ ]:
def softmax(Z):
    """Softmax por filas. Se resta el máximo por estabilidad numérica."""
    Z = np.asarray(Z, dtype=float)
    if Z.ndim == 1:
        Z = Z.reshape(1, -1)
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


# Dos scores cualesquiera
z = np.array([1.3, -0.4])

p_softmax = softmax(z)[0, 0]
p_sigmoide = sigmoide(z[0] - z[1])

print(f"softmax con K=2, clase 0 : {p_softmax:.10f}")
print(f"sigmoide de (z0 - z1)    : {p_sigmoide:.10f}")
print(f"¿Son la misma función?   : {np.isclose(p_softmax, p_sigmoide)}")

> **Consecuencia importante:** al softmax solo le importan las **diferencias**
> entre los scores, no sus valores absolutos. Si le suman 100 a los tres $z_k$,
> las probabilidades no cambian. Guarden esta idea: reaparece en la sección 8
> y es la razón por la que interpretar coeficientes en multinomial es más
> delicado que en el caso binario.

In [ ]:
# El softmax es invariante a sumar una constante a todos los scores
z = np.array([2.0, 0.5, -1.0])

print("scores originales :", z, "->", softmax(z).round(4))
print("scores + 100      :", z + 100, "->", softmax(z + 100).round(4))

### Las cuatro escalas de un clasificador

Esta es la tabla más importante del notebook. En regresión lineal había **un
solo** objeto: $\hat{y}$, en las unidades de la variable respuesta. Aquí hay
**cuatro**, y confundirlas es la fuente principal de errores al leer un modelo
de clasificación:

| # | Escala | Rango | Qué vive ahí | Cómo se pasa a la siguiente |
|---|---|---|---|---|
| 1 | **Logit / score** $z_k$ | $(-\infty, \infty)$ | Los coeficientes $\beta$. Aquí todo **se suma** | $e^{z}$ |
| 2 | **Momios** $e^{z_k}$ | $(0, \infty)$ | Los odds ratios. Aquí todo **se multiplica** | dividir entre la suma |
| 3 | **Probabilidad** $p_k$ | $(0, 1)$ | El softmax, el log-loss | tomar el máximo |
| 4 | **Etiqueta** $\hat{y}$ | $\{A, C, G\}$ | La matriz de confusión, el *accuracy* | — |

Cuando alguien diga "el efecto de la variable", pregunten siempre: **¿en cuál
de las cuatro escalas?** En la sección 4 vamos a recorrer las cuatro con un
pingüino concreto y los números en la mano.

---

## 2. Preparación de los datos

Cargamos exactamente el mismo dataset del notebook de EDA, con la misma
limpieza, para que todo lo que ya observaron siga siendo válido.

In [ ]:
df = sns.load_dataset("penguins").dropna().reset_index(drop=True)

print(f"Registros tras eliminar faltantes: {len(df)}")
print()
print("Distribución de la variable objetivo:")
conteo = df["species"].value_counts()
for especie, n in conteo.items():
    print(f"  {especie:<10} {n:>4}  ({100 * n / len(df):.1f}%)")

**Las clases están desbalanceadas** (44% / 36% / 20%). Eso tiene dos
consecuencias inmediatas:

1. La partición train/test debe ser **estratificada**, para que las tres
   especies aparezcan en la misma proporción en ambos conjuntos. Sin eso, con
   solo 68 Chinstraps, una partición desafortunada puede dejar muy pocos en
   entrenamiento.
2. El **accuracy de referencia no es 33%, es 44%**: un modelo que siempre diga
   "Adelie" y nunca mire los datos acierta 44% de las veces. Cualquier modelo
   que construyamos tiene que ganarle a eso para haber aportado algo.

In [ ]:
VARIABLES = ["bill_length_mm", "bill_depth_mm"]
ETIQUETAS_VAR = ["Longitud del pico (mm)", "Profundidad del pico (mm)"]

X = df[VARIABLES].values
y = df["species"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEMILLA, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} pingüinos")
print(f"Prueba       : {X_test.shape[0]} pingüinos")
print()
print("Proporciones preservadas por la estratificación:")
comparacion = pd.DataFrame(
    {
        "Train": pd.Series(y_train).value_counts(normalize=True).round(3),
        "Test": pd.Series(y_test).value_counts(normalize=True).round(3),
    }
)
print(comparacion.to_string())

### ¿Por qué solo dos variables?

El EDA mostró que **cuatro** medidas distinguen a las especies. Empezamos con
solo dos —longitud y profundidad del pico— por dos razones:

1. **Se pueden dibujar.** Con dos variables la frontera de decisión es visible
   en un plano, y ver la frontera es la mitad de entender el modelo.
2. **Deja errores que estudiar.** Con las cuatro variables el modelo acierta
   *todo* y no hay nada que analizar. Volveremos a eso en la sección 9, porque
   un modelo que no se equivoca nunca es una señal de alarma, no de éxito.

### Estandarización

Igual que en K-Means y en Ridge/Lasso, estandarizamos. Aquí importa por dos
motivos distintos: el descenso en gradiente converge mucho más rápido cuando
las variables están en escalas comparables, y `LogisticRegression` de
`scikit-learn` **aplica regularización L2 por default**, que castiga el tamaño
de los coeficientes y por lo tanto es sensible a las unidades.

Como siempre: el `scaler` se ajusta **solo con entrenamiento** y luego se
*aplica* a prueba. Ajustarlo con todo es una fuga de información.

In [ ]:
scaler = StandardScaler().fit(X_train)

X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print("Aprendido del conjunto de entrenamiento:")
for var, mu, sd in zip(VARIABLES, scaler.mean_, scaler.scale_):
    print(f"  {var:<18} media = {mu:7.3f}   desv. est. = {sd:6.3f}")

print()
print("Tras estandarizar, cada variable tiene media 0 y desviación 1:")
print(f"  medias train : {X_train_s.mean(axis=0).round(6)}")
print(f"  desv.  train : {X_train_s.std(axis=0).round(6)}")

---

## 3. El modelo

Con los datos listos, ajustar el modelo son dos líneas.

Una nota sobre `scikit-learn`: cuando la variable objetivo tiene más de dos
categorías, `LogisticRegression` ajusta **automáticamente** la versión
multinomial con softmax. No hay que pedírselo. (En versiones anteriores existía
un parámetro `multi_class`; fue eliminado en la versión 1.7 justamente porque
el comportamiento multinomial ya es el default.)

In [ ]:
modelo = LogisticRegression(max_iter=1000, random_state=SEMILLA)
modelo.fit(X_train_s, y_train)

print("Clases aprendidas :", modelo.classes_)
print("Forma de coef_    :", modelo.coef_.shape, " <- (n_clases, n_variables)")
print("Forma de intercept_:", modelo.intercept_.shape)

Fíjense en la forma de `coef_`: **(3, 2)**. Una fila por clase, una columna por
variable. En regresión lineal `coef_` era un vector; aquí es una **matriz**,
porque hay un juego completo de coeficientes por cada especie. Eso es
exactamente el "paso 1" del softmax.

In [ ]:
tabla_coef = pd.DataFrame(modelo.coef_, index=modelo.classes_, columns=VARIABLES)
tabla_coef.insert(0, "intercepto", modelo.intercept_)

print("Coeficientes del modelo (sobre variables estandarizadas):")
print(tabla_coef.round(4).to_string())

In [ ]:
acc_train = modelo.score(X_train_s, y_train)
acc_test = modelo.score(X_test_s, y_test)

# Referencia: el modelo que ignora las medidas y solo usa las proporciones
# observadas. Predice siempre la clase mayoritaria, pero reporta como
# probabilidades las frecuencias de cada especie en el entrenamiento.
tonto = DummyClassifier(strategy="prior").fit(X_train_s, y_train)
acc_tonto = tonto.score(X_test_s, y_test)

print(f"Accuracy entrenamiento : {acc_train:.3f}")
print(f"Accuracy prueba        : {acc_test:.3f}")
print(f"Referencia (siempre 'Adelie') : {acc_tonto:.3f}")
print()
print(f"El modelo reduce el error de {1 - acc_tonto:.1%} a {1 - acc_test:.1%}.")

---

## 4. Un pingüino, cuatro escalas

Esta sección es el corazón del notebook. Vamos a tomar **un solo pingüino** del
conjunto de prueba y seguirlo por las cuatro escalas de la tabla de la sección
1, haciendo cada cuenta a mano y verificándola contra `scikit-learn`.

Elegimos deliberadamente el pingüino **más dudoso** del conjunto de prueba: aquel
cuya probabilidad máxima es la más baja. Es el caso más instructivo, porque en
un pingüino fácil todas las cuentas dan 0.999 y no se ve nada.

In [ ]:
probas_test = modelo.predict_proba(X_test_s)

# El pingüino sobre el que el modelo está menos seguro
i = int(np.argmin(probas_test.max(axis=1)))

print(f"Pingüino #{i} del conjunto de prueba")
print(f"  Especie real: {y_test[i]}")
print()
print("  Medidas originales:")
for var, etq, val in zip(VARIABLES, ETIQUETAS_VAR, X_test[i]):
    print(f"    {etq:<28} {val:6.1f}")
print()
print("  Medidas estandarizadas (desviaciones respecto a la media del train):")
for var, val in zip(VARIABLES, X_test_s[i]):
    print(f"    {var:<20} {val:+7.4f}")

### Escala 1: el logit (score)

Un score por especie. Es una suma ponderada, idéntica en forma a la predicción
de una regresión lineal:

$$z_k = \beta_0^k + \beta_1^k \cdot \text{longitud} + \beta_2^k \cdot \text{profundidad}$$

In [ ]:
x_i = X_test_s[i]

z = modelo.intercept_ + modelo.coef_ @ x_i

print("Escala 1 — LOGIT: un score por especie, en (-inf, inf)")
print()
for clase, b0, betas, score in zip(modelo.classes_, modelo.intercept_, modelo.coef_, z):
    detalle = "  +  ".join(
        f"({b:+.3f} x {xv:+.3f})" for b, xv in zip(betas, x_i)
    )
    print(f"  z[{clase:<10}] = {b0:+.3f}  +  {detalle}  =  {score:+.4f}")

### Escala 2: los momios

Exponenciamos. Los scores negativos se vuelven números entre 0 y 1; los
positivos, mayores que 1. **Todo se vuelve positivo y el orden se conserva.**

In [ ]:
momios = np.exp(z)

print("Escala 2 — MOMIOS: e^z, en (0, inf)")
print()
for clase, score, m in zip(modelo.classes_, z, momios):
    print(f"  e^({score:+.4f}) = {m:8.4f}   ({clase})")
print()
print(f"  Suma de los tres: {momios.sum():.4f}")

### Escala 3: la probabilidad

Dividimos cada momio entre la suma. Ahora los tres números suman exactamente 1
y se pueden leer como probabilidades.

In [ ]:
probs_a_mano = momios / momios.sum()
probs_sklearn = modelo.predict_proba(X_test_s[i : i + 1])[0]

print("Escala 3 — PROBABILIDAD: softmax, en (0, 1), suman 1")
print()
print(f"  {'Especie':<12} {'a mano':>10} {'sklearn':>10}")
for clase, pm, ps in zip(modelo.classes_, probs_a_mano, probs_sklearn):
    print(f"  {clase:<12} {pm:10.4f} {ps:10.4f}")
print()
print(f"  Suma: {probs_a_mano.sum():.6f}")
print(f"  ¿Coinciden con sklearn? {np.allclose(probs_a_mano, probs_sklearn)}")

### Escala 4: la etiqueta

Nos quedamos con la especie de mayor probabilidad. **Aquí se pierde
información**, y este pingüino muestra exactamente cuánta.

In [ ]:
etiqueta_a_mano = modelo.classes_[np.argmax(probs_a_mano)]
etiqueta_sklearn = modelo.predict(X_test_s[i : i + 1])[0]

print("Escala 4 — ETIQUETA: argmax")
print()
print(f"  Predicción : {etiqueta_a_mano}   (sklearn: {etiqueta_sklearn})")
print(f"  Realidad   : {y_test[i]}")
print(f"  ¿Acertó?   : {etiqueta_a_mano == y_test[i]}")
print()
print("  Pero la etiqueta esconde esto:")
for clase, p in zip(modelo.classes_, probs_a_mano):
    barra = "#" * int(round(p * 50))
    print(f"    {clase:<12} {p:6.1%}  {barra}")

**Lean las dos últimas celdas juntas.** La matriz de confusión va a registrar
este caso como un error de "Chinstrap clasificado como Gentoo", igual que
registraría uno donde el modelo hubiera dicho Gentoo con 99.9% de seguridad.
Pero no son el mismo error: aquí el modelo estaba **prácticamente en un empate a
tres bandas**, y la diferencia entre la primera y la segunda opción es de unos
pocos puntos porcentuales.

Ese es exactamente el punto de los tres modelos A, B y C del capítulo: la
etiqueta es ciega a la confianza, y la función de costo no puede serlo. Por eso
vamos a evaluar con log-loss además de con *accuracy*.

---

## 5. La frontera de decisión

Con dos variables podemos dibujar lo que el modelo realmente aprendió: una
partición del plano en tres regiones.

In [ ]:
# Malla sobre el espacio estandarizado
h = 0.02
x_min, x_max = X_train_s[:, 0].min() - 0.7, X_train_s[:, 0].max() + 0.7
y_min, y_max = X_train_s[:, 1].min() - 0.7, X_train_s[:, 1].max() + 0.7
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

malla = np.c_[xx.ravel(), yy.ravel()]
Z_pred = modelo.predict(malla)

# Índice numérico de cada clase para poder colorear
codigo = {c: k for k, c in enumerate(modelo.classes_)}
Z_cod = np.array([codigo[c] for c in Z_pred]).reshape(xx.shape)

cmap_regiones = ListedColormap([COLORES_ESPECIE[c] for c in modelo.classes_])

fig, ax = plt.subplots(figsize=(9, 7))
ax.contourf(xx, yy, Z_cod, alpha=0.18, cmap=cmap_regiones, levels=[-0.5, 0.5, 1.5, 2.5])
ax.contour(xx, yy, Z_cod, colors="grey", linewidths=1.2, levels=[0.5, 1.5])

for especie in modelo.classes_:
    m = y_test == especie
    ax.scatter(
        X_test_s[m, 0], X_test_s[m, 1],
        color=COLORES_ESPECIE[especie], label=especie,
        s=55, edgecolors="white", linewidth=0.7, zorder=3,
    )

# Marcar el pingüino dudoso de la sección 4
ax.scatter(
    X_test_s[i, 0], X_test_s[i, 1],
    facecolors="none", edgecolors="black", s=320, linewidth=2.2, zorder=4,
    label="El pingüino dudoso",
)

ax.set_xlabel("Longitud del pico (estandarizada)", fontsize=12)
ax.set_ylabel("Profundidad del pico (estandarizada)", fontsize=12)
ax.set_title(
    "Frontera de decisión de la logística multinomial\n(puntos = conjunto de prueba)",
    fontsize=13,
)
ax.legend(title="Especie real", loc="upper right")
plt.tight_layout()
plt.show()

**Tres cosas que leer en esta gráfica:**

1. **Las fronteras son rectas.** El modelo es lineal *en la escala del logit*, y
   eso se ve: la frontera entre dos clases es el lugar donde sus scores se
   empatan, $z_j = z_k$, que es la ecuación de una recta. La sigmoide y el
   softmax curvan la *probabilidad*, no la frontera. Un modelo que necesite
   fronteras curvas —y lo habrá— pide otra familia de funciones: árboles,
   boosting, redes.
2. **El punto marcado en negro cae casi sobre el vértice** donde se tocan las
   tres regiones. Por eso sus tres probabilidades estaban casi empatadas: no es
   un accidente, es geometría.
3. **Los errores están todos en la frontera Chinstrap–Gentoo.** El EDA ya lo
   anticipaba: las dimensiones del pico separan bien a Adelie, pero Chinstrap y
   Gentoo se traslapan en esa proyección.

### Las probabilidades como superficie

La frontera es solo la línea donde la clase ganadora cambia. Debajo hay una
superficie continua de probabilidad — es la que el modelo realmente estima:

In [ ]:
probas_malla = modelo.predict_proba(malla)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for k, (ax, especie) in enumerate(zip(axes, modelo.classes_)):
    P = probas_malla[:, k].reshape(xx.shape)
    cf = ax.contourf(xx, yy, P, levels=np.linspace(0, 1, 21), cmap="RdYlBu_r", alpha=0.85)
    ax.contour(xx, yy, P, levels=[0.5], colors="black", linewidths=2)

    m = y_test == especie
    ax.scatter(X_test_s[m, 0], X_test_s[m, 1], color="black", s=22, zorder=3)

    ax.set_title(f"P(especie = {especie})", fontsize=12)
    ax.set_xlabel("Long. pico (est.)")
    if k == 0:
        ax.set_ylabel("Prof. pico (est.)")

fig.colorbar(cf, ax=axes, shrink=0.85, label="Probabilidad")
fig.suptitle(
    "Superficie de probabilidad por especie (línea negra = 0.5; puntos = esa especie)",
    fontsize=13, y=1.04,
)
plt.show()

Dos cosas que vale la pena mirar con cuidado:

**La transición de azul a rojo es suave.** Eso es el softmax. La franja
intermedia —donde la probabilidad pasa de 0.2 a 0.8— es angosta, y ahí viven
todos los casos dudosos. Con la etiqueta sola, esa franja es invisible.

**La línea negra de Chinstrap tiene una esquina.** Y no contradice lo que
acabamos de decir de que las fronteras son rectas: son dos cosas distintas.

- La frontera de la gráfica anterior separa *dos clases entre sí*, y es donde se
  empatan sus scores, $z_j = z_k$. Eso siempre es una **recta**.
- La línea de esta gráfica marca dónde $P(\text{Chinstrap}) = 0.5$, que es
  Chinstrap **contra las otras dos juntas**. Para que Chinstrap pase de 0.5 tiene
  que ganarle a Adelie *y* a Gentoo: es la intersección de dos condiciones
  lineales, y la intersección de dos semiplanos tiene una **esquina**.

Adelie y Gentoo no la tienen porque están en los extremos del rango de tamaños;
Chinstrap está en medio, acorralada por las otras dos. Es la misma razón por la
que es la clase difícil del modelo.

---

## 6. El motor, a mano

En el capítulo dijimos que **aprender es mover parámetros para bajar un número**,
y que la regla de actualización de la logística era la misma de la regresión
lineal:

$$\beta_j \leftarrow \beta_j + \frac{\alpha}{n}\sum_{i=1}^{n} r^{(i)} x_j^{(i)}
\qquad \text{con} \qquad r^{(i)} = y^{(i)} - p^{(i)}$$

En el caso multinomial cambia una sola cosa: $y^{(i)}$ ya no es un 0 o un 1, sino
un **vector one-hot** (un 1 en la especie correcta, ceros en las otras dos), y
$p^{(i)}$ es el vector de tres probabilidades del softmax. El residuo
$r^{(i)} = y^{(i)} - p^{(i)}$ sigue significando **exactamente lo mismo**: qué
tan lejos quedó el modelo de la verdad en este ejemplo.

Vamos a programarlo desde cero, en unas quince líneas, y comprobar que llega al
mismo lugar que `scikit-learn`.

In [ ]:
# La variable objetivo, en codificación one-hot
clases = modelo.classes_
Y_train_oh = (y_train[:, None] == clases[None, :]).astype(float)

print("One-hot: cada fila tiene un 1 en la especie correcta")
print()
for j in range(5):
    print(f"  {y_train[j]:<10} -> {Y_train_oh[j]}")

In [ ]:
def entrena_gd(X, Y, alpha=0.5, n_iter=20000, lam=0.0):
    """Descenso en gradiente para logística multinomial.

    lam es la intensidad de la penalización L2 sobre W (no sobre el sesgo).
    Devuelve los pesos, el sesgo y el historial de la función de costo.
    """
    n, p = X.shape
    K = Y.shape[1]

    W = np.zeros((p, K))   # una columna de coeficientes por clase
    b = np.zeros(K)
    historial = []

    for _ in range(n_iter):
        P = softmax(X @ W + b)                        # paso hacia adelante

        log_loss_medio = -np.mean(np.sum(Y * np.log(P + 1e-15), axis=1))
        historial.append(log_loss_medio + lam * np.sum(W**2))

        R = Y - P                                     # el residuo: y - p
        W += alpha * (X.T @ R / n - 2 * lam * W)      # la regla de actualización
        b += alpha * R.mean(axis=0)

    return W, b, np.array(historial)


# Para comparar contra sklearn hay que usar SU objetivo. sklearn minimiza
# 0.5*||W||^2 + C * sum(log-loss), que equivale a  log_loss_medio + ||W||^2/(2*C*n).
C_sklearn = 1.0
n_train = len(y_train)
lam = 1.0 / (2.0 * C_sklearn * n_train)

W_gd, b_gd, historial = entrena_gd(X_train_s, Y_train_oh, alpha=0.5, n_iter=20000, lam=lam)

print(f"Costo inicial (iteración 0)     : {historial[0]:.4f}   <- ln(3) = {np.log(3):.4f}")
print(f"Costo final   (iteración 20000) : {historial[-1]:.4f}")

El costo arranca en $\ln 3 = 1.0986$, que es exactamente lo que vale el log-loss
de un modelo que arranca en cero y por lo tanto asigna 1/3 a cada especie. El
descenso empieza literalmente desde la ignorancia total.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(historial, color="#d62728", linewidth=2)
axes[0].axhline(np.log(3), color="grey", linestyle="--", linewidth=1)
axes[0].text(len(historial) * 0.45, np.log(3) + 0.02, "ln(3): no saber nada", fontsize=9, color="grey")
axes[0].set_xlabel("Iteración")
axes[0].set_ylabel("Función de costo J")
axes[0].set_title("El descenso en gradiente, iteración a iteración")

axes[1].plot(historial[:400], color="#d62728", linewidth=2)
axes[1].set_xlabel("Iteración")
axes[1].set_ylabel("Función de costo J")
axes[1].set_title("Las primeras 400 iteraciones (acercamiento)")

plt.tight_layout()
plt.show()

In [ ]:
coef_gd = W_gd.T   # sklearn guarda (n_clases, n_variables)

comparacion = pd.DataFrame(
    {
        "clase": np.repeat(clases, len(VARIABLES)),
        "variable": VARIABLES * len(clases),
        "a mano (GD)": coef_gd.ravel(),
        "sklearn": modelo.coef_.ravel(),
    }
)
comparacion["diferencia"] = (comparacion["a mano (GD)"] - comparacion["sklearn"]).abs()

print("Coeficientes: descenso en gradiente escrito a mano vs. scikit-learn")
print()
print(comparacion.round(4).to_string(index=False))
print()
print(f"Diferencia máxima en coeficientes : {comparacion['diferencia'].max():.5f}")

P_gd = softmax(X_train_s @ W_gd + b_gd)
P_sk = modelo.predict_proba(X_train_s)
print(f"Diferencia máxima en probabilidad : {np.abs(P_gd - P_sk).max():.6f}")

**Coinciden hasta el tercer decimal.** No es una analogía ni una simplificación:
`scikit-learn` usa un optimizador más sofisticado (L-BFGS, que aprovecha
información de segundo orden para llegar en menos pasos), pero **está resolviendo
exactamente el mismo problema y llega exactamente al mismo lugar**, porque el
log-loss es convexo y tiene un solo mínimo.

Cuando en el capítulo 8 aparezca la frase "entrenar una red neuronal", será este
mismo bucle. La familia de funciones será enorme y habrá que usar la regla de la
cadena para calcular el gradiente —eso es la retropropagación—, pero el ciclo
"predice, mide el error, mueve los parámetros en contra del gradiente, repite"
es el que acaban de escribir.

---

## 7. Evaluación

El *accuracy* de la sección 3 fue nuestro primer número, y es el más pobre de
todos. Ahora lo abrimos.

### Matriz de confusión

In [ ]:
y_pred = modelo.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred, labels=clases)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
ConfusionMatrixDisplay(cm, display_labels=clases).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusión (conjunto de prueba)", fontsize=13)
plt.tight_layout()
plt.show()

print("Con 3 clases la matriz es 3x3: la diagonal son los aciertos,")
print("y cada casilla fuera de ella dice CON QUÉ se confundió el modelo.")
print()
print(f"Total de errores: {(y_pred != y_test).sum()} de {len(y_test)}")

Con dos clases había cuatro casillas y cada una tenía nombre propio (TP, FP, FN,
TN). Con tres clases hay nueve, y los nombres dejan de servir: lo que importa es
**el patrón**. Aquí todos los errores están en una sola casilla, y eso es
información de negocio —o de biología— no de estadística: el modelo no se
equivoca al azar, tiene **una** debilidad concreta.

### Precision y recall por clase

In [ ]:
print(classification_report(y_test, y_pred, digits=3))

**Cómo leer esto con tres clases.** Precision y recall se calculan una clase a la
vez, tratándola como "positiva" y a las otras dos juntas como "negativas":

- **Recall de Chinstrap**: de todos los Chinstrap reales, ¿cuántos detecté?
- **Precision de Gentoo**: de todos los que llamé Gentoo, ¿cuántos lo eran?

Y hay **dos formas de promediar**, que no dan lo mismo:

| Promedio | Cómo | Cuándo usarlo |
|---|---|---|
| `macro avg` | Promedio simple de las tres clases | Todas las clases importan igual, **aunque sean raras** |
| `weighted avg` | Promedio ponderado por el tamaño de cada clase | Interesa el desempeño global de la población |

Con clases desbalanceadas la diferencia importa: **el `macro avg` es más bajo**,
porque los errores se concentran en Chinstrap, que es la clase más pequeña (20%
de los datos) y por lo tanto la que el promedio ponderado casi no ve. Reportar
solo el `weighted avg` esconde justamente el problema que tiene el modelo.

### Log-loss: lo que el accuracy no ve

In [ ]:
probas_train = modelo.predict_proba(X_train_s)
probas_test = modelo.predict_proba(X_test_s)

ll_train = log_loss(y_train, probas_train, labels=clases)
ll_test = log_loss(y_test, probas_test, labels=clases)

# Referencias
ll_ignorante = np.log(3)  # asignar 1/3 a cada clase
probas_tonto = tonto.predict_proba(X_test_s)
ll_tonto = log_loss(y_test, probas_tonto, labels=tonto.classes_)

print(f"{'Modelo':<38} {'Accuracy':>9} {'Log-loss':>10}")
print("-" * 60)
print(f"{'No saber nada (1/3 a cada clase)':<38} {1/3:>9.3f} {ll_ignorante:>10.4f}")
print(f"{'Solo las proporciones del train':<38} {acc_tonto:>9.3f} {ll_tonto:>10.4f}")
print(f"{'Logística multinomial (train)':<38} {acc_train:>9.3f} {ll_train:>10.4f}")
print(f"{'Logística multinomial (test)':<38} {acc_test:>9.3f} {ll_test:>10.4f}")

**Cómo leer esa tabla.** El log-loss es el precio promedio que paga el modelo por
la probabilidad que le asignó a lo que de verdad ocurrió. Se lee al revés que el
accuracy: **más bajo es mejor**, y $\ln 3 = 1.0986$ es la línea de flotación —
lo que paga quien no sabe nada y reparte 1/3 a cada especie.

Fíjense en la segunda fila. Un modelo que solo conoce las proporciones históricas
gana 11 puntos de accuracy sobre el ignorante (0.44 contra 0.33) y sin embargo
**su log-loss es prácticamente el mismo**. Tiene sentido: no aprendió nada sobre
ningún pingüino en particular, solo aprendió a apostarle al más común.

> **Advertencia práctica.** Si ese modelo base reportara etiquetas duras en lugar
> de proporciones —100% Adelie, 0% a las otras dos— su log-loss se dispararía a
> un número enorme, porque $-\ln(0)$ es infinito y las librerías lo recortan a
> un valor grande. Es la misma lección de la tabla del capítulo: **el castigo por
> jurar y equivocarse no tiene techo.** Por eso un clasificador que se usará con
> log-loss nunca debe reportar 0 ni 1 exactos.

### Los errores, con sus probabilidades

Aquí es donde el log-loss se gana el sueldo. Miremos **cómo** se equivocó el
modelo, no solo cuántas veces:

In [ ]:
errores = np.where(y_pred != y_test)[0]

print(f"Los {len(errores)} errores del modelo, con las probabilidades que asignó:")
print()
print(f"{'#':>4} {'Real':<11} {'Predicho':<11} " + " ".join(f"{c:>10}" for c in clases) + "   margen")
print("-" * 78)
for j in errores:
    p = probas_test[j]
    orden = np.sort(p)[::-1]
    margen = orden[0] - orden[1]
    print(
        f"{j:>4} {y_test[j]:<11} {y_pred[j]:<11} "
        + " ".join(f"{v:10.3f}" for v in p)
        + f"   {margen:7.3f}"
    )

print()
p_correcta = probas_test[errores, [list(clases).index(c) for c in y_test[errores]]]
print("Probabilidad que el modelo le dio a la especie CORRECTA en sus errores:")
print(f"  mínima {p_correcta.min():.3f}   máxima {p_correcta.max():.3f}   media {p_correcta.mean():.3f}")

**Este es el resultado más importante de la sección.** Los cuatro errores tienen
un margen pequeño entre la primera y la segunda opción, y en todos ellos el
modelo le dio a la especie correcta una probabilidad **nada despreciable** —
nunca la descartó.

En el lenguaje del capítulo: este modelo **falla dudando**, que es la forma
correcta de fallar. Es el modelo B, no el C. Un modelo con el mismo *accuracy*
de 96% pero que se equivocara diciendo 0.99 sería mucho peor, y **la matriz de
confusión los mostraría idénticos**. Solo el log-loss los distingue.

### Curvas ROC (una por clase)

La ROC se define para dos clases. Con tres, se dibuja una por especie, tratando
cada una contra el resto (*one-vs-rest*):

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for k, especie in enumerate(clases):
    y_bin = (y_test == especie).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, probas_test[:, k])
    auc = roc_auc_score(y_bin, probas_test[:, k])
    ax.plot(fpr, tpr, linewidth=2.3, color=COLORES_ESPECIE[especie],
            label=f"{especie} vs. resto (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Clasificador aleatorio")
ax.set_xlabel("Tasa de falsos positivos", fontsize=12)
ax.set_ylabel("Sensibilidad (recall)", fontsize=12)
ax.set_title("Curvas ROC one-vs-rest", fontsize=13)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

auc_macro = roc_auc_score(y_test, probas_test, multi_class="ovr", average="macro")
print(f"AUC macro (promedio de las tres): {auc_macro:.4f}")

El AUC de Chinstrap es el más bajo de los tres — consistente con todo lo demás:
es la especie que al modelo le cuesta trabajo. Las tres curvas están muy pegadas
a la esquina superior izquierda, lo que anticipa el tema de la sección 9.

---

## 8. Interpretación de coeficientes (y por qué aquí hay una trampa)

En la logística binaria la lectura era limpia: $e^{\beta}$ es el factor por el
que se multiplican los momios. En la multinomial hay una complicación que
**casi todos los tutoriales omiten**, y que viene directamente de la propiedad
que vimos en la sección 1.

Recuerden: al softmax solo le importan las **diferencias** entre scores. Sumarle
una constante a los tres no cambia nada. Eso significa que la matriz de
coeficientes **no es única**: existen infinitos juegos de $\beta$ que producen
exactamente las mismas probabilidades.

`scikit-learn` resuelve la ambigüedad eligiendo la solución **centrada**, donde
los coeficientes de cada variable suman cero entre las clases. Verifiquémoslo:

In [ ]:
print("Suma de cada columna de coef_ (una suma por variable):")
for var, s in zip(VARIABLES, modelo.coef_.sum(axis=0)):
    print(f"  {var:<20} {s:+.10f}")
print()
print("Suman cero. La consecuencia es la siguiente:")

> **Un coeficiente de la multinomial, por sí solo, no significa nada.**
> Solo tienen interpretación las **diferencias entre dos filas**, y esas
> diferencias sí son únicas.

La diferencia entre las filas de dos clases $a$ y $b$ recupera exactamente la
logística binaria entre esas dos clases:

$$\ln\frac{P(y=a)}{P(y=b)} = (\beta_0^a - \beta_0^b) + \sum_j (\beta_j^a - \beta_j^b)\, x_j$$

Es decir: **el log de los momios de $a$ contra $b$ es lineal**, con coeficientes
iguales a la resta de las filas. Ahí sí se puede hablar de odds ratios.

In [ ]:
def compara_clases(a, b):
    ia, ib = list(clases).index(a), list(clases).index(b)
    dif = modelo.coef_[ia] - modelo.coef_[ib]

    print(f"{a} vs. {b}")
    print("  (por cada desviación estándar de aumento en la variable)")
    print()
    for var, etq, d in zip(VARIABLES, ETIQUETAS_VAR, dif):
        factor = np.exp(d)
        if factor >= 1:
            lectura = f"multiplica los momios por {factor:6.2f}"
        else:
            lectura = f"los divide entre {1 / factor:6.2f}"
        print(f"    {etq:<28} beta = {d:+7.3f}   e^beta = {factor:8.3f}   {lectura}")
    print()


for a, b in [("Chinstrap", "Adelie"), ("Gentoo", "Adelie"), ("Chinstrap", "Gentoo")]:
    compara_clases(a, b)

**Cómo se lee la primera comparación.** Una desviación estándar más de longitud
del pico (unos 5.5 mm) multiplica por ~46 los momios de ser Chinstrap en lugar
de Adelie. Es un efecto enorme, y coincide con lo que el EDA ya había mostrado:
la longitud del pico es *la* variable que separa a Chinstrap de Adelie.

La profundidad del pico va en la dirección contraria en esa comparación, pero es
un efecto mucho más chico. Su papel grande está en la tercera comparación,
Chinstrap vs. Gentoo, que es la frontera difícil del modelo.

::: Cuatro advertencias al presentar estos números :::

1. **Los coeficientes están en unidades estandarizadas.** "Una desviación
   estándar" son 5.5 mm de longitud y 2.0 mm de profundidad, no un milímetro.
   Para reportar por milímetro hay que dividir entre `scaler.scale_`.
2. **Un coeficiente suelto de `coef_` no se interpreta.** Solo las diferencias.
   Si alguien reporta "el coeficiente de Gentoo para profundidad es −2.94", está
   reportando un artefacto de la convención de centrado de `sklearn`.
3. **Momios no son probabilidad.** Multiplicar los momios por 46 no multiplica
   la probabilidad por 46. La traducción depende de dónde estaba el pingüino.
4. **Esto no es causalidad.** El modelo midió una asociación morfológica. Nada
   de esto dice qué pasaría si *interviniéramos* sobre el pico de un pingüino.

---

## 9. Cuando el modelo sale demasiado bien

Hasta aquí usamos dos variables. El EDA decía que hay **cuatro** medidas útiles.
Usémoslas todas — y veamos qué pasa.

In [ ]:
TODAS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

X4 = df[TODAS].values
X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4, y, test_size=0.3, random_state=SEMILLA, stratify=y
)
sc4 = StandardScaler().fit(X4_train)
X4_train_s, X4_test_s = sc4.transform(X4_train), sc4.transform(X4_test)

modelo4 = LogisticRegression(max_iter=1000, random_state=SEMILLA)
modelo4.fit(X4_train_s, y4_train)

p4_test = modelo4.predict_proba(X4_test_s)

print(f"Accuracy entrenamiento : {modelo4.score(X4_train_s, y4_train):.4f}")
print(f"Accuracy prueba        : {modelo4.score(X4_test_s, y4_test):.4f}")
print(f"Log-loss prueba        : {log_loss(y4_test, p4_test, labels=modelo4.classes_):.4f}")
print()
print("Matriz de confusión:")
print(pd.DataFrame(
    confusion_matrix(y4_test, modelo4.predict(X4_test_s), labels=modelo4.classes_),
    index=modelo4.classes_, columns=modelo4.classes_,
).to_string())

**100% en el conjunto de prueba. Cero errores.**

La reacción correcta ante este resultado **no es celebrar, es desconfiar.** En
un problema real, un 100% casi siempre significa una de tres cosas:

1. **Fuga de información**: una variable predictora que en realidad contiene la
   respuesta, o que no estará disponible al momento de predecir de verdad.
2. **Contaminación del conjunto de prueba**: el escalador ajustado con todos los
   datos, filas duplicadas entre train y test, etc.
3. **El problema es genuinamente fácil.**

Aquí es la tercera, y hay que decirlo con claridad: **los pingüinos son un
dataset excepcionalmente separable**. Tres especies biológicamente distintas,
medidas con cuidado, sin ruido de medición apreciable. Casi ningún problema de
negocio se parece a esto. Un modelo de churn con 96% de *accuracy* sería
sospechoso; con 100%, seguro está mal.

Miremos qué le pasó a las probabilidades:

In [ ]:
max_2var = probas_test.max(axis=1)
max_4var = p4_test.max(axis=1)

fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.linspace(0.33, 1.0, 40)
ax.hist(max_2var, bins=bins, alpha=0.65, label="Modelo de 2 variables", color="#E69F00")
ax.hist(max_4var, bins=bins, alpha=0.65, label="Modelo de 4 variables", color="#56B4E9")
ax.set_xlabel("Probabilidad de la clase ganadora")
ax.set_ylabel("Número de pingüinos del conjunto de prueba")
ax.set_title("Cuánta duda le queda al modelo")
ax.legend()
plt.tight_layout()
plt.show()

for nombre, mx in [("2 variables", max_2var), ("4 variables", max_4var)]:
    print(f"{nombre:<14} mediana {np.median(mx):.4f}   "
          f"% por encima de 0.99: {100 * (mx > 0.99).mean():5.1f}%   "
          f"mínimo {mx.min():.3f}")

El modelo de 4 variables casi no tiene zona de duda: sus probabilidades se
amontonan contra el 1. Eso es lo que significa **separabilidad** en la práctica,
y es la antesala del problema de la siguiente sección.

---

## 10. Separación perfecta y regularización

El capítulo advertía: cuando una combinación de variables separa perfectamente
las clases, **el log-loss no tiene mínimo**. Siempre se puede mejorar un poquito
más haciendo los coeficientes más grandes, empujando las probabilidades hacia
0 y 1. El optimizador, obediente, se va al infinito.

`LogisticRegression` de `scikit-learn` aplica regularización L2 por default
(`C=1.0`), así que normalmente esto queda oculto. Quitémosla para verlo.

In [ ]:
# C = infinito equivale a "sin penalización"
modelo_sin_reg = LogisticRegression(max_iter=50000, C=np.inf, random_state=SEMILLA)
modelo_sin_reg.fit(X4_train_s, y4_train)

print(f"{'':<24} {'|coef| máx':>11} {'train acc':>10} {'test acc':>9} {'train LL':>10}")
print("-" * 68)
for nombre, mm in [("Con L2 (C=1, default)", modelo4), ("Sin penalización (C=inf)", modelo_sin_reg)]:
    print(
        f"{nombre:<24} {np.abs(mm.coef_).max():11.2f} "
        f"{mm.score(X4_train_s, y4_train):10.3f} {mm.score(X4_test_s, y4_test):9.3f} "
        f"{log_loss(y4_train, mm.predict_proba(X4_train_s), labels=mm.classes_):10.6f}"
    )

Sin penalización el coeficiente más grande pasa de ~2.4 a ~36: **quince veces más
grande, para clasificar exactamente igual de bien.** El log-loss de entrenamiento
baja casi a cero, que era justo lo que el optimizador tenía que lograr. Hizo su
trabajo a la perfección; el modelo resultante es peor.

¿Por qué peor, si acierta igual?

- **Es ininterpretable.** Un coeficiente de 36 sobre variables estandarizadas no
  describe biología, describe una asíntota numérica.
- **Sus probabilidades son mentira.** Reporta 0.99999 donde honestamente debería
  decir 0.95.
- **Es frágil.** Los coeficientes dependen de qué pingüinos cayeron en el
  entrenamiento; con otra partición cambiarían drásticamente. Eso es varianza,
  en el sentido exacto del capítulo.

### Cuándo la regularización sí cambia las predicciones

En este dataset la regularización protege la interpretabilidad, pero no mejora
el *accuracy*: los datos son demasiado limpios. Para ver la otra mitad de la
historia hay que ponerlo en el régimen donde los modelos de verdad viven —
**pocos datos y muchas variables**, la mayoría inútiles.

Simulamos eso: las 4 medidas reales más 25 variables de puro ruido, y solo 60
pingüinos de entrenamiento.

In [ ]:
rng = np.random.default_rng(0)

X_ruido = np.hstack([df[TODAS].values, rng.normal(size=(len(df), 25))])

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_ruido, y, train_size=60, random_state=3, stratify=y
)
scr = StandardScaler().fit(Xr_train)
A, B = scr.transform(Xr_train), scr.transform(Xr_test)

VALORES_C = [np.inf, 100, 10, 1.0, 0.3, 0.1, 0.03, 0.01]
filas = []

for valor_c in VALORES_C:
    mm = LogisticRegression(max_iter=200000, C=valor_c, random_state=SEMILLA).fit(A, yr_train)
    filas.append({
        "C": valor_c,
        "|coef| max": np.abs(mm.coef_).max(),
        "acc train": mm.score(A, yr_train),
        "acc test": mm.score(B, yr_test),
        "log-loss train": log_loss(yr_train, mm.predict_proba(A), labels=mm.classes_),
        "log-loss test": log_loss(yr_test, mm.predict_proba(B), labels=mm.classes_),
    })

resultados = pd.DataFrame(filas)
print("4 variables reales + 25 de ruido, 60 pingüinos de entrenamiento")
print()
print(resultados.round(4).to_string(index=False))

mejor = resultados.loc[resultados["log-loss test"].idxmin()]
print()
print(f"Mejor log-loss de prueba en C = {mejor['C']}")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))

x_pos = np.arange(len(VALORES_C))
ax.plot(x_pos, resultados["log-loss train"], "o-", linewidth=2.3,
        color="#1f77b4", label="Error de entrenamiento")
ax.plot(x_pos, resultados["log-loss test"], "o-", linewidth=2.3,
        color="#ff7f0e", label="Error en datos nuevos (prueba)")

k_mejor = int(resultados["log-loss test"].idxmin())
ax.axvline(k_mejor, color="grey", linestyle="--", linewidth=1.2)
ax.annotate("el punto óptimo",
            xy=(k_mejor, resultados["log-loss test"].iloc[k_mejor]),
            xytext=(k_mejor + 0.6, resultados["log-loss test"].iloc[k_mejor] + 0.18),
            arrowprops={"arrowstyle": "->", "color": "grey"}, fontsize=10, color="grey")

ax.set_xticks(x_pos)
ax.set_xticklabels([str(c) for c in VALORES_C])
ax.set_xlabel("C  (más a la izquierda = menos regularización)", fontsize=12)
ax.set_ylabel("Log-loss", fontsize=12)
ax.set_title("Optimizar no es aprender", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Esta gráfica es la del capítulo, reproducida en vivo.**

Léanla de izquierda a derecha, que es la dirección en la que se regulariza más:

- La curva **azul** (entrenamiento) solo empeora. Sin penalización el modelo
  ajusta el entrenamiento casi perfecto, y cada incremento de regularización lo
  deteriora. Si nuestro criterio fuera el error de entrenamiento, la respuesta
  sería siempre la misma: $C = \infty$, nunca regularizar.
- La curva **naranja** (datos nuevos) baja, toca un mínimo y vuelve a subir.
  Ese mínimo es el punto óptimo, y **está en un lugar completamente distinto**.

La distancia entre los dos mínimos es, literalmente, la diferencia entre
optimizar y aprender.

Y noten el detalle que cierra el círculo del notebook: mientras el log-loss de
prueba se mueve casi 50%, **el `accuracy` de prueba casi no se mueve**. Si
hubiéramos elegido `C` mirando el *accuracy*, no habríamos visto nada y
cualquier valor nos habría parecido igual de bueno. La métrica que ve el
problema es la que castiga la confianza mal calibrada.

Por último: `C` no se puede elegir minimizando el error de entrenamiento, y aquí
lo elegimos mirando el conjunto de prueba — que es **hacer trampa**, porque ese
conjunto ya no sirve como estimación honesta del desempeño. La forma correcta es
la validación cruzada, y es exactamente el tema del capítulo 4.

---

## 11. Resumen

### Lo que cambió respecto a la regresión lineal

| | Regresión lineal | Logística binaria | Logística multinomial |
|---|---|---|---|
| **Predice** | Un número | Una probabilidad | Un vector de $K$ probabilidades |
| **Familia** | $\beta^T x$ | $\sigma(\beta^T x)$ | $\text{softmax}(W^T x)$ |
| **Parámetros** | Un vector | Un vector | Una **matriz** $K \times p$ |
| **Costo** | Error cuadrático | Log-loss | Log-loss (K términos) |
| **Residuo** | $y - \hat{y}$ | $y - p$ | $y_{\text{one-hot}} - p$ |
| **Optimizador** | Fórmula cerrada o GD | GD (no hay fórmula cerrada) | GD (no hay fórmula cerrada) |
| **Evaluación** | RMSE, $R^2$ | Confusión 2×2, ROC, log-loss | Confusión $K \times K$, ROC OvR, log-loss |

Las tres filas de en medio son las que importan: **el residuo y el optimizador
son el mismo en las tres columnas**. Lo único que cambió al pasar de una
regresión lineal a un clasificador de tres clases fue la familia de funciones.

### Los ocho puntos que hay que llevarse

1. El **softmax** es la sigmoide con más de dos clases. Un score por clase,
   exponenciar, normalizar.
2. Hay **cuatro escalas** —logit, momios, probabilidad, etiqueta— y cada
   afirmación sobre un modelo vive en una de ellas. Preguntar siempre en cuál.
3. El paso de probabilidad a etiqueta **destruye información**. Todo lo
   interesante vive antes de ese paso.
4. El *accuracy* es ciego a la confianza; el **log-loss** no. Dos modelos con la
   misma matriz de confusión pueden valer cosas muy distintas.
5. Con clases desbalanceadas, el promedio **macro** y el **ponderado** cuentan
   historias diferentes. Reportar solo el ponderado esconde a las clases chicas.
6. En multinomial, **un coeficiente suelto no se interpreta**: solo las
   diferencias entre clases.
7. Un modelo que acierta el 100% es motivo de **sospecha**, no de celebración.
8. El optimizador minimiza con obediencia perfecta lo que se le pidió. Que eso
   sea aprender, y no solo optimizar, es responsabilidad de quien elige la
   función de costo y cuándo detenerse.

### Qué sigue

- **Capítulo 4 — Evaluación**: cómo elegir `C` sin hacer trampa (validación
  cruzada), y qué métrica usar según el costo del error.
- **Capítulos 5 a 7 — Árboles, bosques y boosting**: qué hacer cuando la
  frontera de decisión no puede ser una recta.
- **Capítulo 8 — Redes neuronales**: el mismo bucle de la sección 6, con una
  familia de funciones mucho más grande. El softmax y el log-loss de este
  notebook siguen ahí, en la última capa.

### Ejercicios

1. Repitan el modelo de la sección 3 con el par `bill_depth_mm` +
   `flipper_length_mm`. El *accuracy* baja a ~83%. Dibujen la frontera de
   decisión y expliquen, con el EDA en la mano, **por qué** ese par es peor.
2. Agreguen `island` al modelo de 4 variables (con `pd.get_dummies`). Comparen
   los coeficientes con `C=1` y con `C=np.inf`. ¿Por qué explotan? Pista: revisen
   la tabla cruzada especie × isla del notebook de EDA.
3. En la sección 6, cambien la tasa de aprendizaje `alpha` a 0.01 y a 5.0.
   Grafiquen las tres curvas de costo juntas. ¿Qué pasa en cada caso?
4. El modelo de 2 variables se equivoca solo con Chinstraps. Si equivocarse en un
   Chinstrap costara diez veces más que en las otras especies, ¿qué cambiarían?
   (Pista: `class_weight` en `LogisticRegression`, y vuelvan a la sección de
   umbral y matriz de costos del capítulo.)